In [1]:
import os
import pickle
from datetime import datetime
from pathlib import Path
from neural_decoder.neural_decoder_trainer import trainModel


In [2]:
# Set parameters directly
dataset_path = "silentspeller_decoder/data/pca64"  # ここにデータセットのパスを設定
model_name = Path(dataset_path).name  # データセットのディレクトリ名をモデル名として使用

print(f"Dataset path: {dataset_path}")
print(f"Model name: {model_name}")


Dataset path: silentspeller_decoder/data/pca64
Model name: pca64


In [3]:
# Load dataset to get input features dimension
with open(dataset_path, "rb") as handle:
    dataset = pickle.load(handle)
n_input_features = dataset['train'][0]['sentenceDat'][0].shape[1]

print(f"Number of input features: {n_input_features}")
print(f"Dataset keys: {list(dataset.keys())}")
print(f"Number of training samples: {len(dataset['train'])}")


Number of input features: 64
Dataset keys: ['train', 'test', 'competition', 'processing_info']
Number of training samples: 1


In [4]:
# データセット名の抽出（パスの最後の部分を使用）
dataset_name = Path(dataset_path).stem

# 日時とデータセット名でログディレクトリ名を生成
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
log_dir_name = f"{timestamp}_{dataset_name}"
if model_name:  # オプションのモデル名が指定されていれば追加
    log_dir_name += f"_{model_name}"

print(f"Dataset name: {dataset_name}")
print(f"Log directory name: {log_dir_name}")


Dataset name: pca64
Log directory name: 20250625_101501_pca64_pca64


In [5]:
model_args = {}
model_args['outputDir'] = os.path.join('./logs', log_dir_name)
model_args['datasetPath'] = dataset_path

# 時系列長パラメータ
model_args['seqLen'] = 250        # 平均長に近い値（245.17）を切り上げ
model_args['maxTimeSeriesLen'] = 512  # 最大長（501）を超える2のべき乗

# トレーニングパラメータ
model_args['batchSize'] = 64
model_args['lrStart'] = 0.02
model_args['lrEnd'] = 0.02
model_args['nUnits'] = 128
model_args['nBatch'] = 10000
model_args['nLayers'] = 5
model_args['seed'] = 0

# モデルアーキテクチャパラメータ
model_args['nClasses'] = 40
model_args['nInputFeatures'] = n_input_features  # データセットから自動的に設定
model_args['dropout'] = 0.4
model_args['strideLen'] = 4
model_args['kernelLen'] = 32
model_args['bidirectional'] = True
model_args['l2_decay'] = 1e-5

# データ拡張パラメータ
model_args['whiteNoiseSD'] = 0.8
model_args['constantOffsetSD'] = 0.2
model_args['gaussianSmoothWidth'] = 2.0

print("Model arguments:")
for key, value in model_args.items():
    print(f"  {key}: {value}")


Model arguments:
  outputDir: ./logs/20250625_101501_pca64_pca64
  datasetPath: silentspeller_decoder/data/pca64
  seqLen: 250
  maxTimeSeriesLen: 512
  batchSize: 64
  lrStart: 0.02
  lrEnd: 0.02
  nUnits: 128
  nBatch: 10000
  nLayers: 5
  seed: 0
  nClasses: 40
  nInputFeatures: 64
  dropout: 0.4
  strideLen: 4
  kernelLen: 32
  bidirectional: True
  l2_decay: 1e-05
  whiteNoiseSD: 0.8
  constantOffsetSD: 0.2
  gaussianSmoothWidth: 2.0


In [6]:
print(f"Starting training...")
print(f"Output directory: {model_args['outputDir']}")
print("="*50)

trainModel(model_args)

print("="*50)
print("Training completed!")


Starting training...
Output directory: ./logs/20250625_101501_pca64_pca64
Using CUDA device


/opt/yj/python310/lib/python3.10/site-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/jupyter/src/neural_decoder/augmentations.py:91: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  return self.conv(input, weight=self.weight, groups=self.groups, padding="same")
/home/jupyter/src/neural_decoder/neural_decoder_trainer.py:187: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(pred[iterIdx, 0 : adjustedLens[iterIdx], :]),


batch 0, ctc loss: 19.243761, cer: 1.000000, time/batch:   0.005
batch 100, ctc loss: 3.063680, cer: 1.000000, time/batch:   0.024
batch 200, ctc loss: 2.421802, cer: 0.952899, time/batch:   0.024
batch 300, ctc loss: 1.680274, cer: 0.750000, time/batch:   0.026
batch 400, ctc loss: 1.258478, cer: 0.474638, time/batch:   0.025
batch 500, ctc loss: 1.030764, cer: 0.373188, time/batch:   0.026
batch 600, ctc loss: 0.910215, cer: 0.268116, time/batch:   0.025
batch 700, ctc loss: 0.803295, cer: 0.260870, time/batch:   0.025
batch 800, ctc loss: 0.699951, cer: 0.246377, time/batch:   0.025
batch 900, ctc loss: 0.701152, cer: 0.242754, time/batch:   0.026
batch 1000, ctc loss: 0.642327, cer: 0.221014, time/batch:   0.025
batch 1100, ctc loss: 0.632727, cer: 0.221014, time/batch:   0.026
batch 1200, ctc loss: 0.636174, cer: 0.213768, time/batch:   0.025
batch 1300, ctc loss: 0.585956, cer: 0.199275, time/batch:   0.025
batch 1400, ctc loss: 0.604556, cer: 0.202899, time/batch:   0.025
batch 

In [ ]:
# 実行環境の確認
import sys
import os
import socket

print("=== 実行環境情報 ===")
print(f"ホスト名: {socket.gethostname()}")
print(f"現在のディレクトリ: {os.getcwd()}")
print(f"Python実行パス: {sys.executable}")
print(f"Python version: {sys.version}")

print("\n=== モジュールの場所 ===")
try:
    import neural_decoder
    print(f"neural_decoder の場所: {neural_decoder.__file__}")
except ImportError as e:
    print(f"neural_decoder import エラー: {e}")

print("\n=== PYTHONPATH ===")
for i, path in enumerate(sys.path):
    print(f"{i}: {path}")


In [ ]:
%%writefile test_module.py
"""
テスト用のカスタムモジュール
このセルの内容が全てtest_module.pyファイルとして保存される
"""

def custom_function():
    """カスタム関数の例"""
    return "LakeTahoe上で作成されたモジュールです！"

class DataProcessor:
    def __init__(self, data):
        self.data = data
    
    def process(self):
        return f"Processing: {self.data}"

# このファイルが直接実行された場合のテスト
if __name__ == "__main__":
    print("test_module.py が作成されました！")


In [ ]:
# 作成されたファイルを確認
import os
print("ファイル一覧:")
for file in os.listdir("."):
    if file.endswith(".py"):
        print(f"  📄 {file}")

# 作成したモジュールをインポートして使用
import test_module

print(f"\n実行結果: {test_module.custom_function()}")

processor = test_module.DataProcessor("sample data")
print(f"処理結果: {processor.process()}")


In [ ]:
# シェルコマンドの実行例
print("=== 基本情報 ===")
!whoami
!pwd
!hostname

print("\n=== ディスク使用量 ===")
!df -h

print("\n=== メモリ使用量 ===")
!free -h

print("\n=== プロセス一覧（一部） ===")
!ps aux | head -10

print("\n=== 環境変数 ===")
!env | grep -E "(CUDA|PYTHON|PATH)" | head -5


In [ ]:
# 複数行のシェルスクリプト実行
%%bash
echo "=== Git情報（もしある場合） ==="
git --version 2>/dev/null || echo "Git not available"

echo -e "\n=== Python packages ==="
pip list | head -10

echo -e "\n=== ディレクトリ構造 ==="
ls -la | head -10

echo -e "\n=== システム情報 ==="
uname -a


In [ ]:
# 開発用ヘルパー関数の定義
import importlib
import os
from datetime import datetime

def quick_edit_and_test(module_name, test_code=""):
    """
    モジュールの編集→リロード→テストを高速化
    """
    print(f"=== {module_name} 開発サイクル ===")
    
    # ファイルが存在するかチェック
    file_path = f"{module_name}.py"
    if os.path.exists(file_path):
        print(f"✅ {file_path} found")
        
        # モジュールをリロード
        try:
            if module_name in globals():
                importlib.reload(globals()[module_name])
                print(f"🔄 {module_name} reloaded")
            else:
                globals()[module_name] = __import__(module_name)
                print(f"📥 {module_name} imported")
        except Exception as e:
            print(f"❌ Import/Reload error: {e}")
            return False
        
        # テストコードがあれば実行
        if test_code:
            print("🧪 Running test...")
            try:
                exec(test_code)
                print("✅ Test passed")
            except Exception as e:
                print(f"❌ Test failed: {e}")
        
        return True
    else:
        print(f"❌ {file_path} not found")
        return False

def live_code_session(module_name):
    """
    Live coding用のセッション開始
    """
    print(f"🎯 Live Coding Session: {module_name}")
    print("=" * 50)
    print("使い方:")
    print(f"1. %%writefile {module_name}.py でコード作成")
    print(f"2. quick_edit_and_test('{module_name}', 'test_code') でテスト")
    print("3. 必要に応じて繰り返し")
    print("=" * 50)

# セッション開始
live_code_session("my_model")


In [ ]:
# 実例: Live Coding風のモジュール開発
%%writefile my_model.py
"""
Live Codingで開発するモデルモジュール
このファイルを編集→テスト→改良のサイクルで開発
"""

import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, input_size=64, hidden_size=128, output_size=40):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x
    
    def get_info(self):
        """モデル情報を返す（デバッグ用）"""
        total_params = sum(p.numel() for p in self.parameters())
        return f"Model with {total_params} parameters"

# バージョン管理用
__version__ = "0.1.0"
print(f"my_model v{__version__} created!")


In [ ]:
# テスト用のコードを定義
test_my_model = """
# my_modelのテスト
model = my_model.MyModel()
print(f"Model info: {model.get_info()}")

# 簡単な動作テスト
x = torch.randn(1, 64)
output = model(x)
print(f"Output shape: {output.shape}")
print(f"Version: {my_model.__version__}")
"""

# モジュールをテスト
quick_edit_and_test("my_model", test_my_model)


In [ ]:
# コード修正の例（バージョンアップ）
%%writefile my_model.py
"""
Live Codingで開発するモデルモジュール v0.2.0
レイヤーを追加してみる
"""

import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, input_size=64, hidden_size=128, output_size=40):
        super().__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)  # 新しいレイヤー追加
        self.fc3 = nn.Linear(hidden_size, output_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)  # 新しいレイヤー通過
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc3(x)
        return x
    
    def get_info(self):
        """モデル情報を返す（デバッグ用）"""
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return f"Model: {total_params} total params, {trainable_params} trainable"

# バージョン管理用
__version__ = "0.2.0"
print(f"my_model v{__version__} updated with extra layer!")


In [ ]:
# 即座にテスト（修正版のテスト）
quick_edit_and_test("my_model", test_my_model)


In [ ]:
# より高度なLive Coding環境の構築
class LiveCodingEnv:
    def __init__(self):
        self.modules = {}
        self.test_suites = {}
        self.version_history = {}
    
    def create_module(self, name, code, test_code=""):
        """モジュール作成と自動テスト"""
        # ファイルに書き込み
        with open(f"{name}.py", "w") as f:
            f.write(code)
        
        # テストコードを保存
        if test_code:
            self.test_suites[name] = test_code
        
        # 即座にテスト
        self.test_module(name)
        print(f"✅ Module '{name}' created and tested")
    
    def update_module(self, name, code):
        """モジュール更新"""
        # バックアップ作成
        if os.path.exists(f"{name}.py"):
            timestamp = datetime.now().strftime("%H%M%S")
            !cp {name}.py {name}_backup_{timestamp}.py
        
        # 新しいコードで更新
        with open(f"{name}.py", "w") as f:
            f.write(code)
        
        # 自動テスト
        self.test_module(name)
        print(f"🔄 Module '{name}' updated")
    
    def test_module(self, name):
        """モジュールのテスト実行"""
        try:
            # リロード
            if name in self.modules:
                importlib.reload(self.modules[name])
            else:
                self.modules[name] = __import__(name)
            
            # テスト実行
            if name in self.test_suites:
                print(f"🧪 Testing {name}...")
                exec(self.test_suites[name], {"modules": self.modules, name: self.modules[name]})
            
            return True
        except Exception as e:
            print(f"❌ Error in {name}: {e}")
            return False
    
    def watch_and_test(self, name, interval=2):
        """ファイル監視（擬似的）"""
        import time
        last_modified = os.path.getmtime(f"{name}.py") if os.path.exists(f"{name}.py") else 0
        
        print(f"👀 Watching {name}.py for changes...")
        for i in range(10):  # 10回まで監視
            time.sleep(interval)
            if os.path.exists(f"{name}.py"):
                current_modified = os.path.getmtime(f"{name}.py")
                if current_modified > last_modified:
                    print(f"🔔 Change detected in {name}.py!")
                    self.test_module(name)
                    last_modified = current_modified

# Live Coding環境をインスタンス化
live_env = LiveCodingEnv()
print("🎯 Live Coding Environment Ready!")
